In [1]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers

In [3]:
import tensorflow_gnn as tfgnn

In [4]:
graph_tensor = tfgnn.GraphTensor.from_pieces(context=tfgnn.Context.from_fields(features={"global_feature": tf.constant([0.5])}),
                                               node_sets={"nodes": tfgnn.NodeSet.from_fields(sizes=[3], 
                                            features={"features": tf.constant([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]], dtype=tf.float32),
                                                   "labels": tf.constant([0, 1, 0], dtype=tf.int32)})},
                                            edge_sets={"edges": tfgnn.EdgeSet.from_fields(
                                                sizes=[2],
                                                adjacency=tfgnn.Adjacency.from_indices(
                                                    source=("nodes", [0, 1]),
                                                    target=("nodes", [1, 2])
                                                ),
                                                features={"edge_features": tf.constant([0,1], dtype=tf.float32)}
                                            )})

In [8]:
class GNNModel(tf.keras.Model):
    def __init__(self, num_classes):
        super().__init__()
        # Dense layer for node feature transformation
        self.dense_transform = layers.Dense(16, activation='relu')
        # Output classification layer
        self.output_layer = layers.Dense(num_classes, activation='softmax')

    def call(self, graph):
        # Transform node features
        node_features = graph.node_sets["nodes"].features["features"]
        transformed = self.dense_transform(node_features)
        # Classification output
        output = self.output_layer(transformed)
        return output

In [9]:
num_classes = 2
model = GNNModel(num_classes)
model.compile(optimizer='adam', 
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False), 
              metrics=['accuracy'])

# Get labels and create dataset
labels = graph_tensor.node_sets["nodes"].features["labels"]
print("Model built successfully!")
print(f"Input shape: {graph_tensor.node_sets['nodes'].features['features'].shape}")
print(f"Labels shape: {labels.shape}")


Model built successfully!
Input shape: (3, 2)
Labels shape: (3,)
Model built successfully!
Input shape: (3, 2)
Labels shape: (3,)


In [10]:
# Test the model
output = model(graph_tensor)
print("Model output shape:", output.shape)
print("Model predictions:\n", output.numpy())

Model output shape: (3, 2)
Model predictions:
 [[0.38570163 0.6142984 ]
 [0.22620416 0.77379584]
 [0.12025984 0.87974024]]


In [12]:
# Check all library versions
print("=" * 60)
print("LIBRARY VERSIONS")
print("=" * 60)

# TensorFlow version
print(f"\nTensorFlow version: {tf.__version__}")

# Keras version
import keras
print(f"Keras version: {keras.__version__}")

# TensorFlow GNN version
print(f"TensorFlow GNN version: {tfgnn.__version__}")

# NumPy version
import numpy as np
print(f"NumPy version: {np.__version__}")

# Matplotlib version
print(f"Matplotlib version: {plt.matplotlib.__version__}")



LIBRARY VERSIONS

TensorFlow version: 2.20.0
Keras version: 3.12.0
TensorFlow GNN version: 1.0.3
NumPy version: 2.2.6
Matplotlib version: 3.10.7
